# Linelist generator

This notebook generates the line lists used by `cloudy` in the grid creation
process, from the reference cloudy models in `stellar_*/` and `agn_*/` (see
the README for how to run them):

- the **standard list**, a reduced list of the lines that are bright enough
  (relative to H$\beta$) in at least one model, at different metallicities;
- the **full list**, every line that Cloudy transfers, built with
  `full_linelist.py`.

In [ ]:
import numpy as np

from syncretize.cloudy import cloudy23

In [ ]:
# define the wavelength type, either standard or just using vacuum wavelengths.
wavelength_type = "standard"
# wavelength_type = 'vacuum'

# define the reference line to use, by default here we use Hbeta
if wavelength_type == "standard":
    reference_line = "H 1 4861.32A"
if wavelength_type == "vacuum":
    reference_line = "H 1 4862.69A"

# log constrast relative to Hbeta to consider, i.e. only include lines
# which are within this value of the reference line.
contrast_stars = -1.5
contrast_agn = -1.5

# minimum wavelength of lines to consider
min_wavelength = 1000

# maximum wavelength of lines to consider
max_wavelength = 25000

### Stellar models

Read in stellar model line lists and identify lines above the threshold.

In [ ]:
linelist = []

model = r"stellar_" + wavelength_type
for i in range(1, 5):
    line_ids, blends, wavelengths, intrinsic, emergent = cloudy23.read_lines(
        rf"{model}/{i}"
    )
    Hbeta = emergent[line_ids == reference_line][0]
    s = (
        (emergent > (Hbeta + contrast_stars))
        & (wavelengths < max_wavelength)
        & (wavelengths > min_wavelength)
        & np.logical_not(blends)
    )
    print(i, np.sum(s))
    linelist += list(line_ids[s])

### AGN models

Read in AGN model line lists and identify lines above the threshold.

In [ ]:
model = r"agn_" + wavelength_type

for i in range(1, 4):
    line_ids, blends, wavelengths, intrinsic, emergent = cloudy23.read_lines(
        rf"{model}/{i}"
    )
    Hbeta = emergent[line_ids == reference_line][0]
    s = (
        (emergent > (Hbeta + contrast_agn))
        & (wavelengths < max_wavelength)
        & (wavelengths > min_wavelength)
        & np.logical_not(blends)
    )
    print(i, np.sum(s))
    linelist += list(line_ids[s])

# Add additional lines
If you want additional lines that might not satisfy the Hbeta
constrast add them in `extra_lines.dat`. They should be picked from 
cloudy list of lines and not be arbitrary.

In [ ]:
with open("extra_lines.dat", "r") as f:
    extra_lines = [line.rstrip() for line in f]
linelist += list(extra_lines)
linelist.sort()
print(linelist)

In [ ]:
linelist = list(set(linelist))
linelist.sort()

print(len(linelist))

linelist_ = []

for line in linelist:
    element, ion, wavelength = line.split(" ")

    if len(ion) == 1:
        line_ = f"{element} {ion} {wavelength}"
        print(line_)
        linelist_.append(line_)

    elif len(ion) == 0:
        line_ = f"{element} {wavelength}"
        print(line_)
        linelist_.append(line_)

with open(f"linelist-{wavelength_type}1.dat", "w") as file:
    file.writelines("\n".join(linelist_) + "\n")

## Full line list

Every line that Cloudy transfers, i.e. all the lines in the line contribution
(`linecont`) of the saved continuum, built by `full_linelist/full_linelist.py`
(its docstring explains what is dropped and why). The Cloudy version is read
from the runs' main output and names the outputs, e.g.
`full_linelist/exclude-c25.00.txt` and `../linelist-full-c25.00.dat` (next to
the other line lists).

The list uses the standard (air) wavelength convention, so only the
`*_standard` models are used.

In [ ]:
from full_linelist.full_linelist import build, check

# Step 1, before running the reference models: the unfiltered list, from the
# labels saved by full_linelist/line_labels.in (sh run.sh line_labels in
# full_linelist/). The models in stellar_standard/ and agn_standard/ save
# their line luminosities for it.
build(
    "full_linelist/line_labels", unfiltered=True, add=["linelist-standard.dat"]
)

In [ ]:
# Step 2, after running the reference models: the labels to exclude
# (ambiguous labels, from the warnings in the .out files, and labels that
# resolve to the same line as another), written to
# full_linelist/exclude-{version}.txt
runs = [f"stellar_standard/{i}" for i in range(1, 5)] + [
    f"agn_standard/{i}" for i in range(1, 4)
]
exclude = check(
    "full_linelist/linelist-unfiltered.dat",
    runs,
    keep=["linelist-standard.dat"],
)

In [ ]:
# Step 3: the full list, ../linelist-full-{version}.dat (next to the other
# line lists)
build(
    "full_linelist/line_labels",
    add=["linelist-standard.dat"],
    exclude=[exclude],
)

## Convert between standard and vacuum wavelengths

In [ ]:
def vacuum_to_air(wave):
    """
    Function to convert vaccum wavelengths to air wavelengths.
    """

    wave2 = wave**2.0

    fact = 1.0 + 2.735182e-4 + 131.4182 / wave2 + 2.76249e8 / (wave2**2.0)

    fact = fact * (wave >= 2000.0) + 1.0 * (wave < 2000.0)

    wave = wave / fact

    return wave


def air_to_vacuum(wave):
    """
    Function to convert air wavelengths to vacuum wavelengths.
    """

    sigma2 = (1.0e4 / wave) ** 2.0  # Convert to wavenumber squared

    # Compute conversion factor

    fact = (
        1.0
        + 6.4328e-5
        + 2.94981e-2 / (146.0 - sigma2)
        + 2.5540e-4 / (41.0 - sigma2)
    )

    fact = fact * (wave >= 2000.0) + 1.0 * (wave < 2000.0)

    wave = wave * fact  # Convert Wavelength

    return wave


air_to_vacuum(vacuum_to_air(4365.0))

In [ ]:
# check this works correctly

standard = open("linelist-standard.dat", "r").readlines()
vacuum = open("linelist-vacuum.dat", "r").readlines()

print(standard)

for standard_, vacuum_ in zip(standard, vacuum):
    # remove escape character
    standard_ = standard_[:-1]
    vacuum_ = vacuum_[:-1]

    element, ion, wavelength = standard_.split(" ")

    wavelength_ = float(wavelength[:-1])
    wavelength_unit = wavelength[-1]

    # convert to vacuum
    new_vacuum = (
        f"{element} {ion}{air_to_vacuum(wavelength_):.6g}{wavelength_unit}"
    )

    print(
        standard_, wavelength_, wavelength_unit, "|", new_vacuum, "|", vacuum_
    )